# Amendment 31: each arm at its own tuned rate; the circuit without the squash

All sessions: **CPU + High-RAM**, notebook **saved in your Drive** (File → Save a copy in Drive), the same commit SHA in cell 1.

| Session | ROLE | What it does | Cells |
|---|---|---|---|
| 1 | `G1` | tunes the VQC and the matched head on raw z (720 runs), then half of the VQC raw-z runs (400) | 1–7, then 8, 9 |
| 2 | `G2` | first the 349 unclipped learning-rate runs of Amendment 30 again, on the right processor (about 30 min), then low-rank k = 0 / k = 8 at the inherited and tuned rates (2,400 runs) | 1–7, then 10a, 10 |
| 3 | `G3` | re-upload Fourier fit at both rates (1,600 runs), then the other half of the VQC raw-z runs (400) and, only if tuning asks for it, the matched head at its new rate (800) | 1–7, then 11, 12 |
| any, when G1–G3 are done | same | the analysis (B = 2000, about 3–4 h) | 13 |
| 4, 5, 6 (overnight) | `H1`, `H2`, `H3` | re-execute every new run on CPU | 1–7, then 14 |
| any, at the end | — | verdict counts | 15 |

**Processor:** every session must be **AMD EPYC 7B12** (all earlier runs were). Cell 5 stops otherwise: then Runtime → Disconnect and delete runtime, connect again, and run from cell 1.

**Rules:** every cell must end with `OK`; if not, stop and send the output. Results go to Drive; finished runs are skipped on a restart (run cells 1–7 again, then the cell you were on). Progress lines show counts and a time estimate from the runs done so far, never a result.

In [ ]:
# 1. SETTINGS. Edit these two lines only.
EXPECTED_SHA = "PASTE_THE_FULL_SHA_OF_THE_AMENDMENT_31_COMMIT"
ROLE = "G1"            # "G1", "G2", "G3", "H1", "H2" or "H3"

BACKUP = "/content/drive/MyDrive/QML-backup-run"
EXP31  = BACKUP + "/exp31"                 # Amendment 26 records (read only)
OUT    = BACKUP + "/cpu_reruns"            # all CPU-check outputs; this notebook writes 35/36/37 here
REPO, ART, ENV = "/content/repo", "/content/art", "/content/env"
PY = ENV + "/bin/python"
DATASETS = ["breastmnist", "pneumoniamnist", "bloodmnist", "pathmnist"]
REGIMES  = ["5", "10", "20", "50", "100"]
TUNE_SEEDS = [90001, 90002, 90003, 90004, 90005]
T35 = T36 = 2                              # threads per process: tuning and raw-z runs (circuits benefit)
T37 = 1                                    # threads per process: the classical rate-gap runs (many processes)
EXPECT = {"35_lr_nosquash": 720, "37_rate_gaps": 4000}

import os, subprocess, json, glob, time, shutil, re
assert ROLE in ("G1", "G2", "G3", "H1", "H2", "H3"), ROLE
os.environ.update({"QML_ARTIFACT_ROOT": ART, "MPLBACKEND": "Agg", "CUDA_VISIBLE_DEVICES": ""})

def sh(cmd, log=None, check=True, env=None):
    """Run a bash command from the repo, print its output, append it to a log on Drive."""
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(["bash", "-lc", cmd], cwd=REPO if os.path.isdir(REPO) else "/content",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=e)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    rc = p.wait()
    if log:
        os.makedirs(OUT + "/logs", exist_ok=True)
        with open(f"{OUT}/logs/{log}", "a") as f:
            f.write(f"\n$ {cmd}\n" + "".join(lines) + f"exit code {rc}\n")
    if check and rc != 0:
        raise RuntimeError(f"FAILED (exit {rc}): {cmd}")
    return "".join(lines)

def launch(cmds, threads, tag):
    """Start commands side by side in the background, each with `threads` threads. Logs on Drive."""
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    procs = {}
    for name, cmd in cmds.items():
        log = f"{OUT}/logs/{tag}_{name}.txt"
        procs[name] = subprocess.Popen(["bash", "-lc", f"{cmd} >> {log} 2>&1"], cwd=REPO,
                                       env=env, start_new_session=True)
        print(f"started {name} (pid {procs[name].pid}, {threads} threads), log: {log}")
    return procs

def count(ns, **where):
    """Shards of a namespace on Drive whose keys match `where` (values compared as text)."""
    n = 0
    for fn in glob.glob(f"{OUT}/shards/{ns}/*.json"):
        if where:
            k = json.load(open(fn))["keys"]
            if any(str(k.get(a)) != str(v) for a, v in where.items()):
                continue
        n += 1
    return n

def run_and_watch(cmds, threads, tag, ns, total, where=None, every=900):
    """Launch, then every 15 minutes: runs done and a time estimate from the rate so far. No results."""
    where = where or {}
    n0, t0 = count(ns, **where), time.time()
    procs = launch(cmds, threads, tag)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every)
        n, dt = count(ns, **where), (time.time() - t0) / 3600
        eta = f", about {dt * (total - n) / (n - n0):.1f} h left at the rate so far" if n > n0 else ""
        print(time.strftime("%H:%M"), f"{ns}: {n} of {total}{eta}", flush=True)
    codes = {nm: p.returncode for nm, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive"
    n = count(ns, **where)
    assert n == total, f"{ns}: {n} of {total} runs"

def verdicts(target):
    """Count replay verdicts in a target namespace on Drive."""
    from collections import Counter
    c = Counter()
    for fn in glob.glob(f"{OUT}/shards/{target}/*.json"):
        try:
            v = json.load(open(fn)).get("replay_check", {}).get("verdict", "?")
        except Exception:
            v = "unreadable"
        c[v.split(" (")[0]] += 1
    return dict(c)

def halves(seq, k):
    return [seq[i::k] for i in range(k)]

def run01(arm, seeds, ns, lr, extra=""):
    """src/01 at one rate, d = 4, every dataset and n, the given seeds."""
    q = f" --lr-quantum {lr}" if arm.startswith("quantum") else ""
    return (f"{PY} -u src/01_frozen_backbone_ablation.py --arms {arm} --dims 4 "
            f"--datasets {' '.join(DATASETS)} --regimes {' '.join(REGIMES)} "
            f"--seeds {' '.join(map(str, seeds))} --lr-head {lr}{q} --experiment {ns} {extra}")

def rates():
    """Stage-1 selections (written by cell 8). Waits for them if another session is still tuning."""
    path = f"{OUT}/nosquash_rates.json"
    while not os.path.exists(path):
        print(time.strftime("%H:%M"), "waiting for G1's tuning (nosquash_rates.json)", flush=True)
        time.sleep(300)
    r = json.load(open(path))
    assert not any(v["edge"] for v in r.values()), "a selection is on the grid edge: stop and send the output"
    return r["quantum_vqc"]["selected"], r["matched_param_fullrank"]["selected"]

print("ROLE", ROLE, "OK")

In [ ]:
# 2. MOUNT DRIVE and check the inputs exist.
from google.colab import drive
drive.mount("/content/drive")
fc = sorted(glob.glob(BACKUP + "/feature_cache_*.tgz"))
assert fc, "feature_cache_*.tgz not found"
n31 = len(glob.glob(EXP31 + "/shards/31_entangling_range/*.json"))
n32 = len(glob.glob(OUT + "/shards/32_squash/*.json"))
ntc = len(glob.glob(OUT + "/tune_cache/*.pt"))
print(f"31_entangling_range {n31} | 32_squash {n32} | tuning cache files on Drive {ntc}")
assert n31 == 2400 and n32 == 1600, "inputs incomplete: send this output"
print("OK")

In [ ]:
# 3. CLONE THE REPO at the pushed commit; check it is clean and carries Amendment 31.
if not os.path.isdir(REPO):
    sh(f"git clone -b feature/journal-expansion https://github.com/anishfelixm/quantum-classical-expressivity {REPO}")
sh(f"git fetch origin && git checkout --detach {EXPECTED_SHA}")
assert sh("git rev-parse HEAD").strip() == EXPECTED_SHA
assert sh("git status --porcelain").strip() == "", "the clone has local changes"
assert "## Amendment 31" in open(REPO + "/docs/analysis_plan.md").read(), "Amendment 31 missing at this commit"
for f in ("scripts/analyse_amendment31.py", "tests/test_unbounded_angles.py"):
    assert os.path.exists(f"{REPO}/{f}"), f
assert "--unbounded-angles" in open(REPO + "/src/01_frozen_backbone_ablation.py").read()
assert "--no-tanh" in open(REPO + "/src/09_lr_selection.py").read()
print("OK")

In [ ]:
# 4. PYTHON ENVIRONMENT: Python 3.10 with requirements.lock.txt (about 5 minutes).
if not os.path.exists(PY):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba")
    sh(f"/content/bin/micromamba create -y -q -p {ENV} -c conda-forge python=3.10 pip")
    sh(f"{PY} -m pip install -q -r requirements.lock.txt pytest --extra-index-url https://download.pytorch.org/whl/cu118")
sh("apt-get -qq install -y pigz > /dev/null", check=False)
sh(f"{PY} -c \"import sys, torch; from importlib.metadata import version as v; print('python', sys.version.split()[0], '| torch', torch.__version__, '| pennylane', v('pennylane'), '| numpy', v('numpy'), '| default threads', torch.get_num_threads())\"",
   log="environment_a31.txt")
print("OK")

In [ ]:
# 5. DEVICE. CPU only; record this session (cores, processor, role).
info = sh(f"{PY} -c \"import sys; sys.path.insert(0,'src'); import config; print(config.DEVICE.type)\"").split()
assert info[-1] == "cpu", "Use the CPU runtime (Runtime -> Change runtime type -> CPU, High-RAM)."
cpu = sh("lscpu | grep 'Model name' | sed 's/.*: *//'", check=False).strip()
cores = os.cpu_count()
with open(OUT + "/logs/sessions.txt", "a") as f:
    f.write(f"{time.strftime('%Y-%m-%d %H:%M')} role={ROLE} cpu='{cpu}' cores={cores} sha={EXPECTED_SHA[:8]}\n")
print(f"cpu='{cpu}'  cores={cores}")
# Every earlier CPU run and re-execution used AMD EPYC 7B12; another processor rounds differently
# (found 9 November: an Intel session diverged at step 1). Stop rather than mix machines.
assert cpu == "AMD EPYC 7B12", ("wrong processor for this study: Runtime -> Disconnect and delete runtime, "
                                "connect again, run from cell 1")
print("OK")

In [ ]:
# 6. GATES: parity (24 head parameters at d = 4), circuit wiring, and the raw-z permission (Amendment 31).
sh(f"{PY} -m pytest -q tests/test_parity.py tests/test_entangling_ranges.py tests/test_unbounded_angles.py",
   log=f"gates_{ROLE}.txt")
print("OK")

In [ ]:
# 7. ARTIFACT FOLDER: links to Drive and the restored feature cache.
def link(sub, ns, target_dir):
    os.makedirs(f"{ART}/{sub}", exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)
    p = f"{ART}/{sub}/{ns}"
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(target_dir, p)

for sub in ("shards", "predictions"):
    link(sub, "31_entangling_range", f"{EXP31}/{sub}/31_entangling_range")
for ns in ("32_squash", "35_lr_nosquash", "36_nosquash", "37_rate_gaps",
           "C_35_lr_nosquash", "C_36_nosquash", "C_37_rate_gaps",
           "34_lr_inherited", "C_34_lr_inherited", "CU_34_lr_inherited",
           "C_31_entangling_range", "CU_31_entangling_range", "C_32_squash", "CU_32_squash"):
    for sub in ("shards", "predictions", "step_norms", "weights"):
        link(sub, ns, f"{OUT}/{sub}/{ns}")

FC = f"{ART}/feature_cache"
if not os.path.islink(FC):
    if os.path.isdir(FC):
        shutil.rmtree(FC)
    pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    if not pts:
        os.makedirs("/content/fc_extract", exist_ok=True)
        sh(f"tar -I pigz -xf '{fc[-1]}' -C /content/fc_extract")
        pts = glob.glob("/content/fc_extract/**/*__r*__s*.pt", recursive=True)
    os.symlink(os.path.dirname(pts[0]), FC)
assert os.path.realpath(FC).startswith("/content/fc_extract")
seeds_all = json.loads(sh(f"{PY} -c \"import sys, json; sys.path.insert(0,'src'); import config; "
                          f"print(json.dumps(config.CONFIRMATORY_SEEDS))\"").strip().splitlines()[-1])
missing = [(d, r, s) for d in DATASETS for r in REGIMES for s in seeds_all
           if not os.path.exists(f"{FC}/{d}__r{r}__s{s}.pt")]
assert not missing, f"{len(missing)} cache files missing; stop and send this output"

# the tuning seeds' caches: the files session C built and saved on 8 November. Never built here.
need = [f"{d}__r{r}__s{s}.pt" for d in DATASETS for r in ("5", "20", "100") for s in TUNE_SEEDS]
for f in need:
    if not os.path.exists(f"{FC}/{f}") and os.path.exists(f"{OUT}/tune_cache/{f}"):
        shutil.copy(f"{OUT}/tune_cache/{f}", f"{FC}/{f}")
miss = [f for f in need if not os.path.exists(f"{FC}/{f}")]
assert not miss, f"{len(miss)} tuning cache files missing, e.g. {miss[:3]}: send this output"
print("cache files present: 800 of 800, tuning 60 of 60")
print("OK")

In [ ]:
# 8. ROLE G1 - STAGE 1: tune the VQC and the matched head on raw z (720 runs), then select each rate.
assert ROLE == "G1"
def tune(arm, seeds):
    return (f"{PY} -u src/09_lr_selection.py --experiment 35_lr_nosquash --arms {arm} --dim 4 --no-tanh "
            f"--seeds {' '.join(map(str, seeds))}")
cmds = {f"vqc_{i}": tune("quantum_vqc", s) for i, s in enumerate(halves(TUNE_SEEDS, 3))}
cmds["matched"] = tune("matched_param_fullrank", TUNE_SEEDS)
run_and_watch(cmds, T35, "tune35", "35_lr_nosquash", EXPECT["35_lr_nosquash"])
sh(f"{PY} -u scripts/analyse_amendment31.py select", log="stats_a31_select.txt")
shutil.copy(f"{ART}/nosquash_rates.json", f"{OUT}/nosquash_rates.json")
rv, rm = rates()
print(f"selected on raw z: VQC {rv}, matched {rm}")
print("OK")

In [ ]:
# 9. ROLE G1 - STAGE 2, first half: the VQC on raw z at its own rate, half the confirmatory seeds (400 runs).
assert ROLE == "G1"
rv, rm = rates()
mine = seeds_all[0::2]
cmds = {f"vqc_{i}": run01("quantum_vqc", s, "36_nosquash", rv, "--no-tanh --unbounded-angles")
        for i, s in enumerate(halves(mine, 4))}
# count only this session's seeds: the other half runs in G3 at the same time
def mine_done():
    return sum(1 for fn in glob.glob(f"{OUT}/shards/36_nosquash/*.json")
               if (lambda k: k["arm"] == "quantum_vqc" and k["seed"] in set(mine))(json.load(open(fn))["keys"]))
procs = launch(cmds, T36, "nosq36_G1")
t0, n0 = time.time(), mine_done()
while any(p.poll() is None for p in procs.values()):
    time.sleep(900)
    n, dt = mine_done(), (time.time() - t0) / 3600
    print(time.strftime("%H:%M"), f"VQC raw z, this half: {n} of 400" +
          (f", about {dt * (400 - n) / (n - n0):.1f} h left" if n > n0 else ""), flush=True)
codes = {nm: p.returncode for nm, p in procs.items()}
print("exit codes:", codes)
assert all(c == 0 for c in codes.values()) and mine_done() == 400, "send this output"
print("OK")

In [ ]:
# 10a. ROLE G2 - AMENDMENT 30, BLOCK C AGAIN: the 349 learning-rate runs without clipping, on AMD EPYC 7B12.
#      The first attempt ran on an Intel session and is kept, unanalysed, in quarantine (plan, second note).
assert ROLE == "G2" and cpu == "AMD EPYC 7B12"
q = f"{OUT}/quarantine/CU_34_lr_inherited_intel_20261109"
for sub in ("shards", "predictions", "step_norms", "weights"):
    src = f"{OUT}/{sub}/CU_34_lr_inherited"
    if os.path.isdir(src) and os.listdir(src) and not os.path.exists(f"{q}/{sub}"):
        os.makedirs(q, exist_ok=True)
        shutil.move(src, f"{q}/{sub}")
        print(f"moved the Intel runs: {sub} -> {q}/{sub}")
    os.makedirs(src, exist_ok=True)
out = sh(f"{PY} scripts/build_unclipped_view.py --list 34_lr_inherited 2>/dev/null")
names = sorted(set(re.findall(r"34_lr_inherited__\S*?\.json", out)))
assert len(names) == 349, f"{len(names)} runs to re-execute, expected 349"
base = f"{PY} -u scripts/replay.py --source 34_lr_inherited --clip inf --code-at 56a09feb"
open("/content/cu34_first.txt", "w").write(names[0] + "\n")
sh(f"OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 {base} --from-list /content/cu34_first.txt", log="replay34u_amd_first.txt")
chunks = {f"p{i}": c for i, c in enumerate(halves(names[1:], 4))}
for nm, c in chunks.items():
    open(f"/content/cu34_{nm}.txt", "w").write("\n".join(c) + "\n")
procs = launch({nm: f"{base} --from-list /content/cu34_{nm}.txt" for nm in chunks}, 2, "replay34u_amd")
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(time.strftime("%H:%M"), "CU_34:", verdicts("CU_34_lr_inherited"), flush=True)
codes = {nm: p.returncode for nm, p in procs.items()}
print("exit codes:", codes)
assert all(c == 0 for c in codes.values()) and sum(verdicts("CU_34_lr_inherited").values()) == 349, "send this output"
# Amendment 30 P1 at step level for all three blocks, then the selection without clipping (P2)
sh(f"{PY} -u scripts/build_unclipped_view.py --check-steps 31_entangling_range 32_squash 34_lr_inherited",
   log="a30_check_steps.txt")
shutil.copy(f"{OUT}/lr_inherited.json", f"{ART}/lr_inherited.json")
sh(f"{PY} -u scripts/build_unclipped_view.py --build 34_lr_inherited", log="a30_views.txt")
sh(f"{PY} -u scripts/analyse_lr_inherited.py --unclipped", log="stats_lr_inherited_unclipped.txt")
shutil.copy(f"{ART}/lr_inherited_unclipped.json", f"{OUT}/lr_inherited_unclipped.json")
print("OK")

In [ ]:
# 10. ROLE G2 - STAGE 3, low-rank: k = 0 at 1e-2, k = 8 at 1e-2 and at 3e-3 (2,400 runs), 1 thread per process.
assert ROLE == "G2"
cmds = {}
for rank, lr in ((0, "1e-2"), (8, "1e-2"), (8, "3e-3")):
    for i, s in enumerate(halves(seeds_all, 2)):
        cmds[f"k{rank}_{lr}_{i}"] = run01("low_rank", s, "37_rate_gaps", lr, f"--head-rank {rank}")
run_and_watch(cmds, T37, "gaps37_lowrank", "37_rate_gaps", 2400, where={"arm": "low_rank"})
print("OK")

In [ ]:
# 11. ROLE G3 - STAGE 3, re-upload Fourier fit at 1e-2 and at 3e-3 (1,600 runs), 1 thread per process.
assert ROLE == "G3"
cmds = {}
for lr in ("1e-2", "3e-3"):
    for i, s in enumerate(halves(seeds_all, 4)):
        cmds[f"fc_{lr}_{i}"] = run01("fourier_rff_r2", s, "37_rate_gaps", lr)
run_and_watch(cmds, T37, "gaps37_fourier", "37_rate_gaps", 1600, where={"arm": "fourier_rff_r2"})
print("OK")

In [ ]:
# 12. ROLE G3 - STAGE 2, second half: the VQC on raw z (400 runs); then, ONLY if stage 1 selected a rate
#     other than 1e-2 for the matched head, the matched head on raw z at that rate (800 runs).
assert ROLE == "G3"
rv, rm = rates()
mine = seeds_all[1::2]
def done(arm, seeds, lr):
    return sum(1 for fn in glob.glob(f"{OUT}/shards/36_nosquash/*.json")
               if (lambda k: k["arm"] == arm and k["seed"] in set(seeds) and k.get("lrh") == f"{float(lr):.0e}")
               (json.load(open(fn))["keys"]))
jobs = [("quantum_vqc", mine, rv, 400)]
if f"{float(rm):.0e}" != "1e-02":
    jobs.append(("matched_param_fullrank", seeds_all, rm, 800))
    print(f"matched head selected {rm} on raw z: its 40-seed runs are needed")
else:
    print("matched head selected 1e-02 on raw z: the Amendment 28 runs are its runs; none needed")
for arm, seeds, lr, total in jobs:
    extra = "--no-tanh --unbounded-angles" if arm.startswith("quantum") else "--no-tanh"
    cmds = {f"{arm[:7]}_{i}": run01(arm, s, "36_nosquash", lr, extra) for i, s in enumerate(halves(seeds, 4))}
    procs = launch(cmds, T36, f"nosq36_G3_{arm[:7]}")
    t0, n0 = time.time(), done(arm, seeds, lr)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(900)
        n, dt = done(arm, seeds, lr), (time.time() - t0) / 3600
        print(time.strftime("%H:%M"), f"{arm} raw z: {n} of {total}" +
              (f", about {dt * (total - n) / (n - n0):.1f} h left" if n > n0 else ""), flush=True)
    codes = {nm: p.returncode for nm, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()) and done(arm, seeds, lr) == total, "send this output"
print("OK")

In [ ]:
# 13. ANY SESSION, after G1 (cell 9), G2 and G3 have all finished: integrity checks, then the analysis.
rv, rm = rates()
n36 = 800 + (0 if f"{float(rm):.0e}" == "1e-02" else 800)
counts = {"35_lr_nosquash": count("35_lr_nosquash"), "36_nosquash": count("36_nosquash"),
          "37_rate_gaps": count("37_rate_gaps")}
print(counts, "expected", {"35_lr_nosquash": 720, "36_nosquash": n36, "37_rate_gaps": 4000})
assert counts == {"35_lr_nosquash": 720, "36_nosquash": n36, "37_rate_gaps": 4000}, "runs incomplete"
for ns in ("36_nosquash", "37_rate_gaps"):
    recs = [json.load(open(f)) for f in glob.glob(f"{OUT}/shards/{ns}/*.json")]
    assert len({r["git_sha"] for r in recs}) == 1, f"more than one commit in {ns}"
    no_pred = [r["keys"] for r in recs if not os.path.exists(
        f"{OUT}/predictions/{ns}/" + os.path.basename(r.get("predictions_file", "none")))]
    assert not no_pred, f"{ns}: {len(no_pred)} runs without predictions"
shutil.copy(f"{OUT}/nosquash_rates.json", f"{ART}/nosquash_rates.json")
out = sh(f"{PY} -u scripts/analyse_amendment31.py contrasts --workers {max(1, cores - 1)}",
         log="stats_amendment31.txt")
assert "SEED-LEVEL FALLBACK IN USE" not in out, "STOP: seed-level fallback in use. Do not use these numbers."
shutil.copy(f"{ART}/amendment31.json", f"{OUT}/amendment31.json")
print("OK")

In [ ]:
# 14. ROLE H1 / H2 / H3 (overnight) - re-execute every Amendment 31 run on CPU, same threads as recorded.
#     H1: 35_lr_nosquash and 36_nosquash. H2: 37 low-rank. H3: 37 Fourier fit.
assert ROLE in ("H1", "H2", "H3")
def base(src):
    return f"{PY} -u scripts/replay.py --source {src} --clip 20 --code-at {EXPECTED_SHA[:8]}"
if ROLE == "H1":
    plan = [("35_lr_nosquash", T35, {f"a{i}": f"--where arm={a} --seeds {' '.join(map(str, s))}"
                                    for i, (a, s) in enumerate([("quantum_vqc", TUNE_SEEDS[:3]),
                                                                 ("quantum_vqc", TUNE_SEEDS[3:]),
                                                                 ("matched_param_fullrank", TUNE_SEEDS)])}),
            ("36_nosquash", T36, {f"s{i}": f"--seeds {' '.join(map(str, s))}" for i, s in enumerate(halves(seeds_all, 4))})]
elif ROLE == "H2":
    plan = [("37_rate_gaps", T37, {f"k{i}": f"--where arm=low_rank --seeds {' '.join(map(str, s))}"
                                   for i, s in enumerate(halves(seeds_all, 6))})]
else:
    plan = [("37_rate_gaps", T37, {f"f{i}": f"--where arm=fourier_rff_r2 --seeds {' '.join(map(str, s))}"
                                   for i, s in enumerate(halves(seeds_all, 8))})]
for src, threads, groups in plan:
    first = list(groups.values())[0]
    sh(f"OMP_NUM_THREADS={threads} MKL_NUM_THREADS={threads} {base(src)} --quick {first}",
       log=f"replay{src[:2]}_quick_{ROLE}.txt")                       # creates the code worktree first
    procs = launch({n: f"{base(src)} {w}" for n, w in groups.items()}, threads, f"replay{src[:2]}_{ROLE}")
    while any(p.poll() is None for p in procs.values()):
        time.sleep(900)
        print(time.strftime("%H:%M"), f"C_{src} verdicts:", verdicts("C_" + src), flush=True)
    codes = {nm: p.returncode for nm, p in procs.items()}
    print("exit codes:", codes)
    assert all(c == 0 for c in codes.values()), "a process failed: see its log on Drive"
print("OK")

In [ ]:
# 15. ANY SESSION, AT THE END: verdict counts of the Amendment 31 re-execution.
for t in ("C_35_lr_nosquash", "C_36_nosquash", "C_37_rate_gaps"):
    print(f"{t:20s} {verdicts(t)}")
print("Expected (Amendment 31 P4): every run IDENTICAL: C_35 720; C_36 800 or 1600; C_37 4000.")
print("OK")